# Shift-Scan v2 (final): accuracy-gap fixes, baselines, benchmarks and hardware, in ONE Kaggle notebook

This notebook implements every improvement from the last review. It reuses your real v1 results as the baseline
(attach the v1 output, i.e. `runs.jsonl`, as a Kaggle *input*; matching runs are paired seed-by-seed with the same data order).

| Improvement | Where it is implemented |
|---|---|
| Ramp quantization in gradually (FP -> ternary) | `QLinear` blend `lam` + `ramp` option, Stage 2 |
| Higher learning rate | Stage 2 probes 2e-3 and 4e-3 (v1 picked 1e-3 at the top edge of its grid) |
| Train longer | Stage 5, one long run of the best config (`long_tokens`) |
| Learned per-group scales (init from `twn`) | `QLinear(learn_scale=True)`, Stage 2 |
| Keep sensitive layers in full precision | Stage 1 per-projection sensitivity scan, auto-selected under a size budget |
| Honest title / multiplication claim | Stage 4 census variants (analytical) and the final checklist |
| Real kernel result | Stage 4, Triton ternary GEMM vs cuBLAS FP16 (measured) |
| 8-bit / 4-bit embeddings + LM head | Stage 6 (PPL effect, measured) + Stage 4 (storage) |
| 4-bit and 8-bit quantized student baselines | Stage 5, `v2_int8`, `v2_int4` |
| HellaSwag, ARC-easy, PIQA | Stage 6, own log-likelihood evaluator (best effort, reported if a dataset is unavailable) |
| Second model size | optional: set `MODEL_SIZE = "370m"` (see config) |
| Paired statistics, confidence intervals, code hash per run | Stage 7 |

**How decisions are made:** Stage 2 runs short single-seed ablations, keeps only components that lower distillation KD by at least 2%,
stacks them, and checks the stack against the best single variant. The chosen config is then confirmed on 3 seeds at the same 4M tokens as v1.
Nothing is assumed to help in advance; a component that does not help is dropped and the report says so.

**Honest limits:** I could not run this notebook myself (no GPU in my sandbox). Syntax and name checks passed; the core scan math was verified
against a reference recurrence in v1. The first things to watch are the `[gpu]` memory lines and the Gate 1 checks.

**Kaggle settings:** GPU accelerator, Internet ON. Use *Restart & Run All* (the allocator setting needs a fresh process).
Stages run in priority order and a time guard stops launching new runs after `MAX_RUNTIME_H`; finished runs are saved in `runs.jsonl` and skipped on re-run.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # needs a fresh process
import sys, json, time, math, copy, hashlib, platform, subprocess, inspect, random, itertools, warnings, gc, re
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

from packaging import version as _v
if _v.parse(transformers.__version__) < _v.parse("4.44.0"):
    _pip("-U", "transformers")
    import importlib; importlib.reload(transformers)
for _m in ("datasets", "scipy", "matplotlib", "tabulate"):
    try:
        __import__(_m)
    except ImportError:
        _pip(_m)
import datasets, scipy.stats as sst
import matplotlib
import matplotlib.pyplot as plt

# ----------------------------- CONFIG --------------------------------------
CODE_VERSION = "shiftscan-kaggle-v2-final"
PRESET = "lean"              # "smoke" (wiring check, ~40 min) | "lean" (default) | "full" (longer long-run)
MODEL_SIZE = "130m"          # "130m" = main study | "370m" = optional scale check (needs best_config.json from a 130m run)
MODEL_ID_OVERRIDE = None

PRESETS = {
    "smoke": dict(tokens=300_000,   abl_tokens=150_000,   long_tokens=600_000,    seeds=[0],       eval_tokens=32_768,
                  wiki_tokens=32_768,  lambada_n=100,  long_len=2048, long_n=2, tokens_per_step=8_192,  bench_n=40,  scale_tokens=200_000),
    "lean":  dict(tokens=4_000_000, abl_tokens=1_000_000, long_tokens=12_000_000, seeds=[0, 1, 2], eval_tokens=262_144,
                  wiki_tokens=262_144, lambada_n=1000, long_len=8192, long_n=8, tokens_per_step=16_384, bench_n=500, scale_tokens=2_000_000),
    "full":  dict(tokens=4_000_000, abl_tokens=1_000_000, long_tokens=24_000_000, seeds=[0, 1, 2], eval_tokens=262_144,
                  wiki_tokens=262_144, lambada_n=1000, long_len=8192, long_n=8, tokens_per_step=16_384, bench_n=1000, scale_tokens=4_000_000),
}
P = PRESETS[PRESET]
TOKENS, SEEDS = P["tokens"], P["seeds"]          # TOKENS=4M matches your v1 runs, so they pair seed-by-seed
ABL_TOKENS, LONG_TOKENS = P["abl_tokens"], P["long_tokens"]
BENCH_N, SCALE_TOKENS = P["bench_n"], P["scale_tokens"]
SCALE_MODE = MODEL_SIZE != "130m"

# switch stages off here to save GPU hours (they run in this priority order)
DO = dict(sens=True, ablate=True, final=True, full_decay=True, hardware=True, long=True,
          int_baselines=True, plain_ckpt=True, bench=True, embquant=True)

SEQ, MICRO_BS = 512, 2
TOKENS_PER_STEP = P["tokens_per_step"]
N_CALIB = 131_072
GROUP, KMAX, HARPQ_H, CHUNK = 128, 15, 128, 64
ALPHA_STATE = 1.0
FP_ROWS_DEFAULT = "dt_bc"
TRAIN_EMBED = False
BASE_RULE = "twn"            # winner of your v1 ternary-only comparison (absmean 62 / actaware 50 / twn 44 PPL)
BEST_RULE = BASE_RULE
RULES = ["absmean", "twn", "actaware", "twn_act"]
DECAY_MODES = ["nearest", "harpq", "dyadic2"]
FULL_DECAYS = ["nearest", "dyadic2"]     # harpq dropped: it matched nearest in retention error and post-hoc PPL
ACT_STATS = None             # activation-aware rules are not used in v2
LR_BASE, LR_TRY = 1e-3, [2e-3, 4e-3]
RAMP_FRAC = 0.2              # fraction of steps used to blend FP -> quantized
FP_BUDGET = 0.15             # max fraction of ternary parameters kept FP by the sensitivity rule
IMPROVE_MIN = 0.02           # a component must lower short-run KD by >= 2% to be kept
INT_LR = 3e-4                # untuned LR for the int8/int4 QAT baselines (documented in the report)
MAX_RUNTIME_H = 11.0

DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/kaggle/working/shift_scan_v2_results") if Path("/kaggle/working").exists() else Path("./shift_scan_v2_results")
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
def time_left_h(): return MAX_RUNTIME_H - (time.time() - T0) / 3600
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

ENV = dict(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version(),
           device=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"), cuda=torch.version.cuda,
           preset=PRESET, code_version=CODE_VERSION, model_size=MODEL_SIZE)
print(json.dumps(ENV, indent=1))
assert torch.cuda.is_available(), "Turn on a GPU accelerator in Kaggle."

# ---- resume + v1 baseline: read every runs.jsonl found in this dir or attached inputs ----
RUNS_PATH = OUT / "runs.jsonl"
DONE = {}
def _load_jsonl(p):
    for line in Path(p).read_text().splitlines():
        if line.strip():
            r = json.loads(line); DONE.setdefault(r["key"], r)
for p_ in (list(Path("/kaggle/input").glob("**/runs.jsonl")) if Path("/kaggle/input").exists() else []):
    _load_jsonl(p_)
if RUNS_PATH.exists(): _load_jsonl(RUNS_PATH)
_v1 = sorted({r["name"] for r in DONE.values() if r["name"].startswith(("fp_student", "tern_"))})
print(f"Loaded {len(DONE)} earlier run records. v1 baseline configs found: {_v1 if _v1 else 'NONE (attach the v1 output as an input for paired comparisons)'}")

# ---- GPU-hour estimate (assumes the ~1,600 tok/s your T4 measured in v1; 370m ~450) ----
_rate = 450 if SCALE_MODE else 1600
def _h(tok, n=1, ev=0.04): return n * (tok / _rate / 3600 + ev)
_est = [("Stage 1 sensitivity scan", 0.15 if DO["sens"] else 0), ("Stage 2 short ablations (7 runs)", _h(ABL_TOKENS, 7, 0.03) if DO["ablate"] else 0),
        (f"Stage 3 best config x {len(SEEDS)} seeds", _h(TOKENS, len(SEEDS)) if DO["final"] else 0),
        (f"Stage 3 best + decay ({len(FULL_DECAYS)} runs)", _h(TOKENS, len(FULL_DECAYS)) if DO["full_decay"] else 0),
        ("Stage 5 long run", _h(LONG_TOKENS, 1, 0.1) if DO["long"] else 0), ("Stage 5 int8 + int4 baselines", _h(TOKENS, 2) if DO["int_baselines"] else 0),
        ("Stage 5 plain-twn checkpoint (bench baseline)", _h(TOKENS, 1) if DO["plain_ckpt"] else 0), ("Stage 4/6 hardware, benchmarks, emb-quant", 0.8)]
if not SCALE_MODE:
    print("\nEstimated GPU hours (stages run in this order):")
    for n_, h_ in _est: print(f"  {n_:<50s} {h_:5.2f} h")
    print(f"  {'TOTAL':<50s} {sum(h for _, h in _est):5.2f} h   (time guard: {MAX_RUNTIME_H} h; Kaggle quota ~30 h/week, check yours)")

## Phase 0: data, teacher, tokenizer
Same streams as v1 (eval / calibration / train are disjoint and the train prefix is identical), so v1 runs are valid baselines.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_IDS = {"130m": ["AntonV/mamba2-130m-hf"], "370m": ["AntonV/mamba2-370m-hf"]}
SS_IDS = {"130m": "state-spaces/mamba2-130m", "370m": "state-spaces/mamba2-370m"}

def convert_state_spaces(size):
    from huggingface_hub import hf_hub_download
    from transformers import Mamba2Config, Mamba2ForCausalLM
    cj = json.load(open(hf_hub_download(SS_IDS[size], "config.json")))
    d, nl = cj["d_model"], cj["n_layer"]
    sd = torch.load(hf_hub_download(SS_IDS[size], "pytorch_model.bin"), map_location="cpu")
    sd = {k.replace("backbone.embedding.", "backbone.embeddings."): v for k, v in sd.items()}
    cfg = Mamba2Config(vocab_size=sd["backbone.embeddings.weight"].shape[0], hidden_size=d, num_hidden_layers=nl,
                       num_heads=(2 * d) // 64, head_dim=64, state_size=128, n_groups=1, expand=2, conv_kernel=4,
                       chunk_size=256, tie_word_embeddings=True)
    m = Mamba2ForCausalLM(cfg)
    print("convert load_state_dict:", m.load_state_dict(sd, strict=False))
    return f"{SS_IDS[size]} (converted)", m

def load_teacher_hf():
    errs = []
    for c in ([MODEL_ID_OVERRIDE] if MODEL_ID_OVERRIDE else HF_IDS[MODEL_SIZE]):
        try:
            try:
                return c, AutoModelForCausalLM.from_pretrained(c, dtype=torch.float32)
            except TypeError:
                return c, AutoModelForCausalLM.from_pretrained(c, torch_dtype=torch.float32)
        except Exception as e:
            errs.append((c, repr(e)[:200]))
    try:
        return convert_state_spaces(MODEL_SIZE)
    except Exception as e:
        errs.append(("convert", repr(e)[:300]))
    raise RuntimeError(f"Could not load Mamba-2 {MODEL_SIZE}. Errors: {errs}. Set MODEL_ID_OVERRIDE.")

MODEL_ID, BASE = load_teacher_hf()
BASE.eval()
CFG = BASE.config
print("model:", MODEL_ID, "| layers", CFG.num_hidden_layers, "| heads", CFG.num_heads, "| vocab", CFG.vocab_size)

TOK = None
for tid in ([MODEL_ID] if "/" in str(MODEL_ID) and "converted" not in MODEL_ID else []) + ["EleutherAI/gpt-neox-20b"]:
    try:
        TOK = AutoTokenizer.from_pretrained(tid); break
    except Exception as e:
        print("tokenizer fail", tid, repr(e)[:100])
assert TOK is not None
assert len(TOK) <= CFG.vocab_size, "tokenizer larger than model vocab"

def build_token_cache(n_eval, n_calib, n_train, path):
    if path.exists():
        return torch.load(path)
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    need = n_eval + n_calib + n_train
    eos = TOK.eos_token_id if TOK.eos_token_id is not None else 0
    buf, total, batch = [], 0, []
    for ex in ds:
        batch.append(ex["text"])
        if len(batch) == 256:
            for ids in TOK(batch, add_special_tokens=False)["input_ids"]:
                buf.append(torch.tensor(ids + [eos], dtype=torch.int32)); total += len(ids) + 1
            batch = []
            if total >= need: break
    flat = torch.cat(buf)[:need]
    d = dict(eval=flat[:n_eval], calib=flat[n_eval:n_eval + n_calib], train=flat[n_eval + n_calib:])
    torch.save(d, path); return d

_need_train = max(TOKENS, LONG_TOKENS if DO["long"] else 0, SCALE_TOKENS if SCALE_MODE else 0) + SEQ * 64
CACHE = build_token_cache(P["eval_tokens"], N_CALIB, _need_train, OUT / f"tokens_v2_{PRESET}_{MODEL_SIZE}.pt")
EVAL_TOK, CALIB_TOK, TRAIN_TOK = CACHE["eval"], CACHE["calib"], CACHE["train"]
print({k: len(v) for k, v in CACHE.items()})

# WikiText-2 and LAMBADA (best effort; missing ones are reported, never faked)
WIKI_TOK, LAMBADA_TEXTS = None, None
try:
    wt = None
    for _rid in ("Salesforce/wikitext", "wikitext"):
        try:
            wt = load_dataset(_rid, "wikitext-2-raw-v1", split="test"); break
        except Exception as _e:
            print("wikitext source failed:", _rid, repr(_e)[:80])
    assert wt is not None
    ids = TOK("\n\n".join(wt["text"]), add_special_tokens=False)["input_ids"][:P["wiki_tokens"]]
    WIKI_TOK = torch.tensor(ids, dtype=torch.int32)
except Exception as e:
    print("WikiText unavailable:", repr(e)[:120])
try:
    lb = load_dataset("EleutherAI/lambada_openai", "default", split="test")
    LAMBADA_TEXTS = lb["text"][:P["lambada_n"]]
except Exception as e:
    print("LAMBADA unavailable:", repr(e)[:120])

## Core: scan, decay quantizers, quantization-aware layers
New vs v1: `QLinear` supports (a) ternary with optional **learned group scales**, (b) int4/int8 for baselines, (c) a **ramp** `lam` that blends FP -> quantized,
(d) whole projections kept FP (`fp_proj`, from the sensitivity scan).

In [ ]:
def segsum(x):
    T = x.size(-1)
    xc = torch.cumsum(x, -1)
    s = xc[..., :, None] - xc[..., None, :]
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
    return s.masked_fill(~mask, float("-inf"))

def ssd_scan(X, dtA, Bm, Cm, chunk):
    """X (b,L,H,P) already scaled by dt; dtA (b,L,H) log-decay; Bm,Cm (b,L,H,N). Returns (b,L,H,P)."""
    b, L, H, Pd = X.shape
    N = Bm.shape[-1]
    pad = (-L) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad)); dtA = F.pad(dtA, (0, 0, 0, pad))
        Bm = F.pad(Bm, (0, 0, 0, 0, 0, pad)); Cm = F.pad(Cm, (0, 0, 0, 0, 0, pad))
    Lp = L + pad; c = Lp // chunk
    X = X.reshape(b, c, chunk, H, Pd); Bm = Bm.reshape(b, c, chunk, H, N); Cm = Cm.reshape(b, c, chunk, H, N)
    A = dtA.reshape(b, c, chunk, H).permute(0, 3, 1, 2)            # (b,H,c,l)
    Acs = torch.cumsum(A, -1)
    Lmat = torch.exp(segsum(A))                                     # (b,H,c,l,l)
    CB = torch.einsum("bclhn,bcshn->bhcls", Cm, Bm)
    Ydiag = torch.einsum("bhcls,bcshp->bclhp", CB * Lmat, X)
    dstates = torch.exp(Acs[..., -1:] - Acs)                        # (b,H,c,l)
    Bw = Bm * dstates.permute(0, 2, 3, 1)[..., None]
    states = torch.einsum("bclhn,bclhp->bchpn", Bw, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], 1)
    dchunk = torch.exp(segsum(F.pad(Acs[..., -1], (1, 0))))          # (b,H,c+1,c+1)
    states = torch.einsum("bhzc,bchpn->bzhpn", dchunk, states)[:, :-1]
    Cw = Cm * torch.exp(Acs).permute(0, 2, 3, 1)[..., None]
    Yoff = torch.einsum("bclhn,bchpn->bclhp", Cw, states)
    return (Ydiag + Yoff).reshape(b, Lp, H, Pd)[:, :L]

# ---------------------- decay quantizers (log-domain, STE) ------------------
_NEG_LOG_MIN = -math.log(1e-6)
def dyadic2_values(kmax):
    vals = set(2.0 ** -k for k in range(kmax + 1))
    for k1 in range(kmax + 1):
        for k2 in range(k1 + 1, kmax + 1):
            vals.add(2.0 ** -k1 - 2.0 ** -k2)
            s = 2.0 ** -k1 + 2.0 ** -k2
            if s <= 1.0: vals.add(s)
    return np.array(sorted(v for v in vals if 0 < v <= 1.0))

_GRID_CACHE = {}
def _dy_grid(kmax, device):
    key = (kmax, str(device))
    if key not in _GRID_CACHE:
        v = dyadic2_values(kmax)
        nl = np.minimum(-np.log1p(-np.minimum(v, 1 - 1e-12)), _NEG_LOG_MIN)   # -log(a_q)
        nl[v >= 1.0] = _NEG_LOG_MIN
        _GRID_CACHE[key] = torch.tensor(nl, dtype=torch.float32, device=device)  # ascending
    return _GRID_CACHE[key]

def quantize_decay(dtA, mode, kmax=KMAX, H=HARPQ_H):
    """dtA = log a <= 0. Returns log(a_q) with straight-through gradient."""
    if mode is None: return dtA
    d = dtA.detach().float().clamp(max=-1e-9)
    q = (-torch.expm1(d)).clamp(1e-12, 1.0)                  # q = 1 - a
    if mode == "nearest":
        k = torch.round(-torch.log2(q)).clamp(0, kmax)
        qh = torch.pow(2.0, -k)
    elif mode == "harpq":
        lags = torch.unique(torch.logspace(0, math.log10(H), 16, device=d.device).round()).float()
        kf = torch.floor(-torch.log2(q)).clamp(0, kmax)
        kc = (kf + 1).clamp(max=kmax)
        ref = torch.exp(d[..., None] * lags)                   # a^r
        errs = []
        for kk in (kf, kc):
            ah = (1 - torch.pow(2.0, -kk)).clamp_min(1e-6)
            errs.append((ref - torch.exp(lags * torch.log(ah)[..., None])).abs().sum(-1))
        qh = torch.where(errs[0] <= errs[1], torch.pow(2.0, -kf), torch.pow(2.0, -kc))
    elif mode == "dyadic2":
        grid = _dy_grid(kmax, d.device)
        nl = (-d).clamp(1e-9, _NEG_LOG_MIN)
        idx = torch.searchsorted(grid, nl.contiguous()).clamp(1, len(grid) - 1)
        lo, hi = grid[idx - 1], grid[idx]
        pick = torch.where((nl - lo).abs() <= (hi - nl).abs(), lo, hi)
        la = -pick
        return dtA + (la - dtA).detach()
    else:
        raise ValueError(mode)
    la = torch.log((1 - qh).clamp_min(1e-6))
    return dtA + (la - dtA).detach()

# ---------------------- ternary quantization --------------------------------
def ternary_parts(W, rule, g, act_s=None):
    out, inn = W.shape
    Wg = W.reshape(out, inn // g, g)
    aW = Wg.abs()
    if rule in ("absmean", "actaware"):
        delta = aW.mean(-1, keepdim=True)
        q = (Wg / delta.clamp_min(1e-8)).round().clamp(-1, 1)
    elif rule in ("twn", "twn_act"):
        thr = 0.7 * aW.mean(-1, keepdim=True)
        q = torch.sign(Wg) * (aW > thr).to(W.dtype)
    else:
        raise ValueError(rule)
    m = (q != 0).to(W.dtype)
    if rule == "absmean":
        alpha = delta
    elif rule == "twn":
        alpha = (aW * m).sum(-1, keepdim=True) / m.sum(-1, keepdim=True).clamp_min(1)
    else:
        if act_s is None: raise ValueError(f"{rule} needs activation scales")
        s2 = (act_s.to(W.device).float() ** 2).reshape(1, inn // g, g)
        alpha = (s2 * aW * m).sum(-1, keepdim=True) / (s2 * m).sum(-1, keepdim=True).clamp_min(1e-12)
    return q, alpha

def ternary_dense(W, rule, g, act_s=None):
    q, alpha = ternary_parts(W, rule, g, act_s)
    return (q * alpha).reshape(W.shape)


RAMP = {"lam": 1.0}          # 0 = full precision weights, 1 = fully quantized (set by the training loop)

def int_quant(W, bits, g):
    out, inn = W.shape
    Wg = W.reshape(out, inn // g, g)
    qmax = 2 ** (bits - 1) - 1
    s = (Wg.abs().amax(-1, keepdim=True) / qmax).clamp_min(1e-8)
    return ((Wg / s).round().clamp(-qmax, qmax) * s).reshape(out, inn)

class QLinear(nn.Module):
    """QAT linear. kind='ternary' (rule, optional learned group scales) or kind='int' (bits). STE + ramp blend."""
    def __init__(self, lin, kind, group, rule=None, bits=None, fp_mask=None, act_s=None, learn_scale=False):
        super().__init__()
        self.weight, self.bias = lin.weight, lin.bias
        self.kind, self.rule, self.bits, self.group = kind, rule, bits, group
        self.register_buffer("fp_mask", fp_mask, persistent=False)
        self.register_buffer("act_s", act_s, persistent=False)
        ls = None
        if learn_scale and kind == "ternary":
            out, inn = lin.weight.shape
            ls = nn.Parameter(torch.zeros(out, inn // group, 1))
        self.register_parameter("log_scale", ls)
    def forward(self, x):
        W = self.weight.float()
        lam = RAMP["lam"]
        if self.kind == "ternary":
            with torch.no_grad():
                q, alpha = ternary_parts(W.detach(), self.rule, self.group, self.act_s)
            a = alpha if self.log_scale is None else alpha * torch.exp(self.log_scale)   # grad flows to log_scale
            Wq = (q * a).reshape(W.shape)
        else:
            with torch.no_grad():
                Wq = int_quant(W.detach(), self.bits, self.group)
        w = (1 - lam) * W + lam * (Wq + W - W.detach())          # forward = blend; grad to W is identity (STE)
        if self.fp_mask is not None:
            w = torch.where(self.fp_mask[:, None], W, w)
        return F.linear(x, w, self.bias)

class ShiftScanMixer(nn.Module):
    """HF Mamba2Mixer forward (no cache) with optional quantized decay / ternary or int projections."""
    def __init__(self, orig, cfg, qcfg, act_stats):
        super().__init__()
        self.H, self.Pd, self.G, self.N = cfg.num_heads, cfg.head_dim, cfg.n_groups, cfg.state_size
        self.inter = self.H * self.Pd
        self.conv_dim = self.inter + 2 * self.G * self.N
        self.layer_idx = getattr(orig, "layer_idx", 0)
        self.chunk = CHUNK
        self.decay = qcfg.get("decay")
        self.cap = None
        tsl = getattr(orig, "time_step_limit", getattr(cfg, "time_step_limit", (0.0, float("inf"))))
        self.tsl = (float(tsl[0]), float(tsl[1]))
        self.act = getattr(orig, "act", F.silu)
        self.conv1d, self.norm = orig.conv1d, orig.norm
        self.A_log, self.D, self.dt_bias = orig.A_log, orig.D, orig.dt_bias
        self.in_proj, self.out_proj = orig.in_proj, orig.out_proj
        rule, bits = qcfg.get("ternary"), qcfg.get("intq")
        fpset = set(qcfg.get("fp_proj", []))
        if rule or bits:
            g = qcfg.get("group", GROUP)
            learn = bool(qcfg.get("learn_scale", False))
            mask = None
            if rule:
                mask = torch.zeros(orig.in_proj.weight.shape[0], dtype=torch.bool)
                fr = qcfg.get("fp_rows", FP_ROWS_DEFAULT)
                if fr in ("dt", "dt_bc"): mask[-self.H:] = True
                if fr == "dt_bc": mask[2 * self.inter: 2 * self.inter + 2 * self.G * self.N] = True
            kind = "ternary" if rule else "int"
            si = act_stats.get((self.layer_idx, "in")) if act_stats else None
            so = act_stats.get((self.layer_idx, "out")) if act_stats else None
            if f"{self.layer_idx}:in" not in fpset:
                self.in_proj = QLinear(orig.in_proj, kind, g, rule, bits, mask, si, learn)
            if f"{self.layer_idx}:out" not in fpset:
                self.out_proj = QLinear(orig.out_proj, kind, g, rule, bits, None, so, learn)

    def forward(self, hidden_states, *args, **kwargs):
        if self.training and torch.is_grad_enabled():          # activation checkpointing in training
            from torch.utils.checkpoint import checkpoint
            return checkpoint(self._core, hidden_states, use_reentrant=False)
        return self._core(hidden_states)

    def _core(self, hidden_states):
        b, L, _ = hidden_states.shape
        proj = self.in_proj(hidden_states)
        I = self.inter
        d_mlp = (proj.shape[-1] - 2 * I - 2 * self.G * self.N - self.H) // 2
        o = 2 * d_mlp
        gate = proj[..., o:o + I]
        xBC = proj[..., o + I:o + I + self.conv_dim]
        dt = proj[..., o + I + self.conv_dim:]
        xBC = self.act(self.conv1d(xBC.transpose(1, 2))[..., :L].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [I, self.G * self.N, self.G * self.N], -1)
        with torch.autocast("cuda", enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias.float()).clamp(self.tsl[0], self.tsl[1])
            dtA = dt * (-torch.exp(self.A_log.float()))
            if self.cap is not None:
                self.cap["dtA"] = dtA.detach().float().cpu()
            if self.decay:
                dtA = quantize_decay(dtA, self.decay)
            rep = self.H // self.G
            xh = x.float().reshape(b, L, self.H, self.Pd)
            Bh = Bm.float().reshape(b, L, self.G, self.N).repeat_interleave(rep, 2)
            Ch = Cm.float().reshape(b, L, self.G, self.N).repeat_interleave(rep, 2)
            if self.cap is not None and "fx" in self.cap:
                self.cap.update(dt=dt.detach().cpu(), xh=xh.detach().cpu(), Bh=Bh.detach().cpu(),
                                Ch=Ch.detach().cpu(), dtA_q=dtA.detach().cpu())
            y = ssd_scan(xh * dt[..., None], dtA, Bh, Ch, self.chunk) + xh * self.D.float()[None, None, :, None]
        y = self.norm(y.reshape(b, L, I), gate)
        return self.out_proj(y)

def make_model(base, qcfg, act_stats=None):
    m = copy.deepcopy(base)
    for name, mod in list(m.named_modules()):
        if mod.__class__.__name__ == "Mamba2Mixer":
            parent_name, child = name.rsplit(".", 1)
            setattr(m.get_submodule(parent_name), child, ShiftScanMixer(mod, m.config, qcfg, act_stats))
    return m

FP_CFG = dict(ternary=None, decay=None)

## Gate 1: reference equivalence + teacher sanity

In [ ]:
def to_dev(m): return m.to(DEV)

TEACHER = to_dev(make_model(BASE, FP_CFG)).eval()
for p_ in TEACHER.parameters(): p_.requires_grad_(False)

@torch.no_grad()
def eval_ppl(model, tokens, seq=SEQ, bs=2, max_seqs=None):
    model.eval()
    n = len(tokens) // seq
    if max_seqs: n = min(n, max_seqs)
    x = tokens[:n * seq].reshape(n, seq).long()
    nll, cnt = 0.0, 0
    for i in range(0, n, bs):
        xb = x[i:i + bs].to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=xb, use_cache=False).logits
        nll += F.cross_entropy(lg[:, :-1].float().reshape(-1, lg.shape[-1]), xb[:, 1:].reshape(-1), reduction="sum").item()
        cnt += xb[:, 1:].numel()
    ce = nll / cnt
    return math.exp(ce), ce

# --- equivalence of my forward vs original HF forward (fp32, no quantization) ---
orig_gpu = copy.deepcopy(BASE).to(DEV).eval()
xb = EVAL_TOK[:2 * SEQ].reshape(2, SEQ).long().to(DEV)
mine32 = to_dev(make_model(BASE, FP_CFG)).eval()
with torch.no_grad():
    a = orig_gpu(input_ids=xb, use_cache=False).logits.float()
    b_ = mine32(input_ids=xb, use_cache=False).logits.float()
rel = ((a - b_).abs().max() / a.abs().max()).item()
print(f"[Gate 1a] max|logits_HF - logits_mine| / max|logits| = {rel:.3e}")
assert rel < 2e-2, "My forward does not reproduce the HF Mamba-2 forward; check config attribute names / version."
del orig_gpu, mine32, a, b_, xb
gc.collect(); torch.cuda.empty_cache()

T_PPL, T_CE = eval_ppl(TEACHER, EVAL_TOK)
print(f"[Gate 1b] teacher FineWeb-Edu PPL (seq {SEQ}) = {T_PPL:.3f}  (your earlier teacher runs: 24.5 at seq 1024)")
assert T_PPL < 60, "Teacher PPL is far too high: wrong checkpoint/tokenizer or broken load."

## Training engine (distillation from the frozen teacher) with ramp, learned scales, checkpoints

In [ ]:
@torch.no_grad()
def eval_kd_state(model, tokens, n_seq=64):
    model.eval(); x = tokens[:n_seq * SEQ].reshape(n_seq, SEQ).long()
    kd = st = 0.0; nb = 0
    for i in range(0, n_seq, 4):
        xb = x[i:i + 4].to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            to = TEACHER(input_ids=xb, use_cache=False, output_hidden_states=True)
            so = model(input_ids=xb, use_cache=False, output_hidden_states=True)
        V = to.logits.shape[-1]
        kd += F.kl_div(F.log_softmax(so.logits.float().reshape(-1, V), -1), F.softmax(to.logits.float().reshape(-1, V), -1),
                       reduction="batchmean").item()
        st += state_loss(so.hidden_states, to.hidden_states).item(); nb += 1
    return kd / nb, st / nb

def state_loss(hs_s, hs_t):
    tot = 0.0
    for a, b in zip(hs_s[1:], hs_t[1:]):
        b = b.float(); tot = tot + ((a.float() - b) ** 2).mean() / b.var().clamp_min(1e-6)
    return tot / (len(hs_s) - 1)

@torch.no_grad()
def eval_longctx(model, long_tok):
    """Per-position NLL, bucketed. long_tok: (n, L). Returns np.array of bucket means."""
    model.eval(); L = long_tok.shape[1]
    edges = [e for e in (1, 512, 1024, 2048, 4096, 8192) if e < L] + [L]
    tot = torch.zeros(L - 1, dtype=torch.float64)
    for i in range(long_tok.shape[0]):
        xb = long_tok[i:i + 1].long().to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=xb, use_cache=False).logits
        nll = F.cross_entropy(lg[0, :-1].float(), xb[0, 1:], reduction="none").double().cpu()
        tot += nll
    tot /= long_tok.shape[0]
    return [float(tot[a - 1:b - 1].mean()) for a, b in zip(edges[:-1], edges[1:])], edges

@torch.no_grad()
def eval_lambada(model, texts):
    if texts is None: return None
    model.eval(); ok = 0
    for t in texts:
        ctx, last = t.rsplit(" ", 1)
        ci = TOK(ctx, add_special_tokens=False)["input_ids"][-1000:]
        ti = TOK(" " + last, add_special_tokens=False)["input_ids"]
        ids = torch.tensor([ci + ti], device=DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=ids, use_cache=False).logits[0]
        pred = lg[len(ci) - 1: len(ci) + len(ti) - 1].argmax(-1).cpu()
        ok += int(torch.equal(pred, torch.tensor(ti)))
    return ok / len(texts)

LONG_L = P["long_len"]
_n_long = min(P["long_n"], len(EVAL_TOK) // LONG_L)
LONG_TOK = EVAL_TOK[-_n_long * LONG_L:].reshape(_n_long, LONG_L) if _n_long > 0 else None
print("long-context eval sequences:", None if LONG_TOK is None else tuple(LONG_TOK.shape),
      "(concatenated documents: tests state stability, not document-level reasoning)")

TEACHER_METRICS = dict(
    fineweb_ppl=T_PPL, fineweb_ce=T_CE,
    wiki_ppl=(eval_ppl(TEACHER, WIKI_TOK)[0] if WIKI_TOK is not None else None),
    lambada_acc=eval_lambada(TEACHER, LAMBADA_TEXTS),
)
if LONG_TOK is not None:
    TEACHER_METRICS["long_nll"], LONG_EDGES = eval_longctx(TEACHER, LONG_TOK)
print("teacher metrics:", {k: v for k, v in TEACHER_METRICS.items() if k != "long_nll"})
(OUT / "teacher_metrics.json").write_text(json.dumps(TEACHER_METRICS, indent=1))

def src_hash(*objs):
    h = hashlib.sha256()
    for o in objs:
        try: h.update(inspect.getsource(o).encode())
        except Exception: h.update(repr(o).encode())
    return h.hexdigest()[:12]
CODE_HASH = src_hash(ssd_scan, quantize_decay, ternary_parts, int_quant, QLinear, ShiftScanMixer, state_loss, make_model)
print("code hash:", CODE_HASH, "| code version:", CODE_VERSION)

def report_gpu(tag, warn_gb=2.5, top=8):
    a_ = torch.cuda.memory_allocated() / 2**30
    print(f"[gpu] {tag}: allocated {a_:.2f} GB, reserved {torch.cuda.memory_reserved() / 2**30:.2f} GB")
    if a_ > warn_gb:
        seen = []
        for o in gc.get_objects():
            try:
                if torch.is_tensor(o) and o.is_cuda:
                    seen.append((o.numel() * o.element_size(), tuple(o.shape), str(o.dtype)))
            except Exception:
                pass
        seen.sort(reverse=True)
        print("  largest live CUDA tensors (MB, shape, dtype):", [(round(sz / 2**20), sh, dt) for sz, sh, dt in seen[:top]])


def run_experiment(name, qcfg, seed, tokens, lr, full_eval=True, save_ckpt=False):
    key = f"{name}|seed{seed}|tok{tokens}|lr{lr:g}"
    if key in DONE:
        return DONE[key]
    if time_left_h() < 0.3:
        print("time budget exhausted, skipping", key); return None
    set_seed(seed)
    gc.collect(); torch.cuda.empty_cache(); report_gpu(f"before {key}")
    RAMP["lam"] = 1.0
    student = to_dev(make_model(BASE, qcfg, ACT_STATS))
    if not TRAIN_EMBED:
        for n_, p_ in student.named_parameters():
            if "embeddings" in n_ or "lm_head" in n_: p_.requires_grad_(False)
    params = [p_ for p_ in student.parameters() if p_.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    steps = max(1, tokens // TOKENS_PER_STEP)
    accum = max(1, TOKENS_PER_STEP // (MICRO_BS * SEQ))
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.cuda.amp.GradScaler()
    n_seq = min(len(TRAIN_TOK) // SEQ, (tokens + SEQ * 64) // SEQ)       # same data window as v1 for tokens=4M
    data = TRAIN_TOK[: n_seq * SEQ].reshape(n_seq, SEQ)
    perm = torch.randperm(n_seq, generator=torch.Generator().manual_seed(1000 + seed))
    ptr = 0
    init_ppl = eval_ppl(student, EVAL_TOK, max_seqs=64)[0]                # fully quantized, before training
    ramp = float(qcfg.get("ramp", 0.0))
    curve, diverged = [], False
    t_start = time.time(); seen = 0
    for step in range(steps):
        student.train()
        if ramp > 0:
            xr = min(1.0, (step + 1) / max(1.0, ramp * steps)); RAMP["lam"] = 0.5 * (1 - math.cos(math.pi * xr))
        else:
            RAMP["lam"] = 1.0
        kd_acc = st_acc = 0.0
        for _ in range(accum):
            idx = perm[(ptr + torch.arange(MICRO_BS)) % n_seq]; ptr += MICRO_BS
            xb = data[idx].long().to(DEV)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                to = TEACHER(input_ids=xb, use_cache=False, output_hidden_states=True)
            with torch.autocast("cuda", dtype=torch.float16):
                so = student(input_ids=xb, use_cache=False, output_hidden_states=True)
            V = so.logits.shape[-1]
            kd = F.kl_div(F.log_softmax(so.logits.float().reshape(-1, V), -1),
                          F.softmax(to.logits.float().reshape(-1, V), -1), reduction="batchmean")
            st = state_loss(so.hidden_states, to.hidden_states)
            loss = (kd + ALPHA_STATE * st) / accum
            scaler.scale(loss).backward()
            kd_acc += kd.item() / accum; st_acc += st.item() / accum; seen += xb.numel()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if not math.isfinite(kd_acc):
            diverged = True; break
        curve.append((step, kd_acc, st_acc))
        if step % max(1, steps // 10) == 0:
            print(f"  [{name} s{seed}] step {step}/{steps} lam={RAMP['lam']:.2f} kd={kd_acc:.4f} state={st_acc:.4f}")
    RAMP["lam"] = 1.0
    train_time = time.time() - t_start
    rec = dict(key=key, name=name, seed=seed, tokens=tokens, lr=lr, qcfg=qcfg, diverged=diverged,
               tokens_seen=seen, tok_per_s=seen / max(train_time, 1e-9), train_s=train_time, init_ppl=init_ppl,
               loss_curve=curve[:: max(1, len(curve) // 100)] if curve else [],
               env=ENV, code_hash=CODE_HASH, preset=PRESET, model_id=MODEL_ID, ts=time.strftime("%Y-%m-%d %H:%M:%S"))
    if save_ckpt and not diverged:
        pth = OUT / f"ckpt_{name}_s{seed}_t{tokens}.pt"
        torch.save({k: v.detach().cpu() for k, v in student.state_dict().items()}, pth)
        rec["ckpt"] = str(pth)
    if not diverged:
        ppl, ce = eval_ppl(student, EVAL_TOK)
        kdv, stv = eval_kd_state(student, EVAL_TOK)
        m = dict(fineweb_ppl=ppl, fineweb_ce=ce, kd=kdv, state_mse=stv)
        if full_eval:
            m["wiki_ppl"] = eval_ppl(student, WIKI_TOK)[0] if WIKI_TOK is not None else None
            m["lambada_acc"] = eval_lambada(student, LAMBADA_TEXTS)
            if LONG_TOK is not None:
                ln, _ = eval_longctx(student, LONG_TOK)
                m["long_nll"] = ln
                m["long_gap"] = [a - b for a, b in zip(ln, TEACHER_METRICS["long_nll"])]
        rec["metrics"] = m
    else:
        rec["metrics"] = {}
    with open(RUNS_PATH, "a") as f: f.write(json.dumps(rec) + "\n")
    DONE[key] = rec
    print(f"  -> {key}: ppl={rec['metrics'].get('fineweb_ppl')} (teacher {T_PPL:.2f}) "
          f"init_ppl={init_ppl:.1f} {rec['tok_per_s']:.0f} tok/s, {train_time/60:.1f} min")
    del student, opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()
    return rec

def kd_of(rec):
    return rec["metrics"].get("kd", float("inf")) if (rec and rec.get("metrics")) else float("inf")

def load_student(qcfg, ckpt_path):
    m = to_dev(make_model(BASE, qcfg, ACT_STATS))
    sd = torch.load(ckpt_path, map_location="cpu")
    res = m.load_state_dict(sd, strict=False)
    miss = [k for k in res.missing_keys if "lm_head" not in k]
    assert not miss and not res.unexpected_keys, (miss[:3], res.unexpected_keys[:3])
    return m.eval()

## Stage 1: per-projection sensitivity scan (no training)
Ternarize **one** projection at a time (everything else FP) and measure the post-hoc PPL damage. Projections with the largest damage are kept in
full precision, up to `FP_BUDGET` of the ternary parameters. *Measured.*

In [ ]:
def proj_params(which):
    H, Pd, G, N, D = CFG.num_heads, CFG.head_dim, CFG.n_groups, CFG.state_size, CFG.hidden_size
    I = H * Pd
    fp_r = {"none": 0, "dt": H, "dt_bc": H + 2 * G * N}[FP_ROWS_DEFAULT]
    return (2 * I + 2 * G * N + H - fp_r) * D if which == "in" else D * I

SENS_FP, SENS_DF = [], pd.DataFrame()
if DO["sens"] and not SCALE_MODE:
    n_layers = CFG.num_hidden_layers
    allp = [f"{j}:{w}" for j in range(n_layers) for w in ("in", "out")]
    base_ppl = eval_ppl(TEACHER, EVAL_TOK, max_seqs=64)[0]
    rows = []
    for li in range(n_layers):
        for which in ("in", "out"):
            mdl = to_dev(make_model(BASE, dict(ternary=BASE_RULE, decay=None, fp_proj=[p_ for p_ in allp if p_ != f"{li}:{which}"]))).eval()
            ppl = eval_ppl(mdl, EVAL_TOK, max_seqs=64)[0]
            rows.append(dict(layer=li, proj=which, ppl=ppl, delta_logppl=math.log(ppl / base_ppl), params=proj_params(which)))
            del mdl; torch.cuda.empty_cache()
    SENS_DF = pd.DataFrame(rows).sort_values("delta_logppl", ascending=False)
    total = sum(proj_params(w) for _ in range(n_layers) for w in ("in", "out"))
    used = 0
    for _, r in SENS_DF.iterrows():
        if used + r.params <= FP_BUDGET * total:
            SENS_FP.append(f"{int(r.layer)}:{r.proj}"); used += r.params
    SENS_DF.to_csv(OUT / "stage1_sensitivity.csv", index=False)
    print("[MEASURED] most sensitive projections (post-hoc dlogPPL when ONLY this one is ternary):")
    print(SENS_DF.head(10).round(4).to_string(index=False))
    print(f"\nKept FP under a {FP_BUDGET:.0%} budget ({used/total:.1%} of ternary params): {SENS_FP}")
else:
    print("Stage 1 skipped.")

## Stage 2: short single-seed ablation of the improvements
Each candidate is a 1M-token run (same seed, same data). A component is kept only if it lowers eval-KD by at least `IMPROVE_MIN`; kept components are stacked
and the stack is compared against the best single variant. The winner is saved to `best_config.json`.

In [ ]:
def load_best_config():
    for p_ in [OUT / "best_config.json"] + (list(Path("/kaggle/input").glob("**/best_config.json")) if Path("/kaggle/input").exists() else []):
        if Path(p_).exists():
            d = json.loads(Path(p_).read_text()); return d["qcfg"], d["lr"]
    return None

BASE_Q = dict(ternary=BASE_RULE, decay=None)
BEST_Q, BEST_LR, ABL = dict(BASE_Q, ramp=RAMP_FRAC, learn_scale=True), LR_BASE, {}     # fallback if ablation cannot run

if SCALE_MODE:
    cfg_loaded = load_best_config()
    if cfg_loaded: BEST_Q, BEST_LR = cfg_loaded
    print("SCALE MODE: using", "loaded best_config.json" if cfg_loaded else "FALLBACK config (ramp + learned scale): no best_config.json found", BEST_Q, BEST_LR)
elif DO["ablate"]:
    cands = {"abl_base": (dict(BASE_Q), LR_BASE), "abl_ramp": (dict(BASE_Q, ramp=RAMP_FRAC), LR_BASE),
             "abl_lscale": (dict(BASE_Q, learn_scale=True), LR_BASE)}
    for lr_ in LR_TRY: cands[f"abl_lr{lr_:g}"] = (dict(BASE_Q), lr_)
    if SENS_FP: cands["abl_fpsens"] = (dict(BASE_Q, fp_proj=SENS_FP), LR_BASE)
    for n_, (q_, lr_) in cands.items():
        ABL[n_] = run_experiment(n_, q_, 0, ABL_TOKENS, lr_, full_eval=False)
    base_kd = kd_of(ABL["abl_base"])
    imp = {}
    for flag, nm in (("ramp", "abl_ramp"), ("learn_scale", "abl_lscale"), ("fp_proj", "abl_fpsens")):
        if nm in cands and kd_of(ABL[nm]) < base_kd * (1 - IMPROVE_MIN):
            imp[flag] = cands[nm][0][flag]
    lr_names = ["abl_base"] + [f"abl_lr{l:g}" for l in LR_TRY]
    best_lr_name = min(lr_names, key=lambda n: kd_of(ABL[n]))
    comb_lr = cands[best_lr_name][1]
    comb_q = dict(BASE_Q, **imp)
    ABL["abl_combined"] = run_experiment("abl_combined", comb_q, 0, ABL_TOKENS, comb_lr, full_eval=False)
    singles = {n: kd_of(r) for n, r in ABL.items() if n != "abl_combined"}
    best_single = min(singles, key=singles.get)
    if np.isfinite(kd_of(ABL["abl_combined"])) and kd_of(ABL["abl_combined"]) <= singles[best_single]:
        BEST_Q, BEST_LR = comb_q, comb_lr
    elif np.isfinite(singles[best_single]):
        BEST_Q, BEST_LR = cands[best_single]
    ABL_DF = pd.DataFrame([dict(variant=n, kd=kd_of(r), ppl=(r["metrics"].get("fineweb_ppl") if r and r.get("metrics") else None),
                                init_ppl=(r["init_ppl"] if r else None), kd_vs_base=kd_of(r) / base_kd) for n, r in ABL.items()])
    print("\n[MEASURED] short-run ablation (seed 0, %d tokens, lower KD is better):" % ABL_TOKENS)
    print(ABL_DF.round(4).to_string(index=False))
    print("\nkept components:", imp, "| best LR:", comb_lr, "| chosen config:", BEST_Q, "lr", BEST_LR)
    ABL_DF.to_csv(OUT / "stage2_ablation.csv", index=False)
else:
    cfg_loaded = load_best_config()
    if cfg_loaded: BEST_Q, BEST_LR = cfg_loaded
(OUT / "best_config.json").write_text(json.dumps(dict(qcfg=BEST_Q, lr=BEST_LR, tokens=TOKENS, model_size=MODEL_SIZE, code=CODE_VERSION), indent=1))
print("\nBEST CONFIG:", BEST_Q, "lr", BEST_LR)

## Stage 3: confirm the chosen config on matched seeds, then add power-of-two decay
`v2_best` uses the same seeds, tokens and data order as your v1 `tern_twn` runs, so the improvement is a paired comparison.

In [ ]:
if SCALE_MODE:
    run_experiment(f"scale_{MODEL_SIZE}_plain", dict(ternary=BASE_RULE, decay=None), SEEDS[0], SCALE_TOKENS, LR_BASE)
    run_experiment(f"scale_{MODEL_SIZE}_best", BEST_Q, SEEDS[0], SCALE_TOKENS, BEST_LR)
else:
    if DO["final"]:
        for s_ in SEEDS:
            run_experiment("v2_best", BEST_Q, s_, TOKENS, BEST_LR, save_ckpt=(s_ == SEEDS[0]))
    if DO["full_decay"]:
        for m_ in FULL_DECAYS:
            run_experiment(f"v2_best_{m_}", dict(BEST_Q, decay=m_), SEEDS[0], TOKENS, BEST_LR)
print(f"time used so far: {(time.time()-T0)/3600:.2f} h")

## Stage 4: hardware evidence (cheap, runs before the long training)
Fixed-point shift-scan fidelity (measured), real packed storage and the Triton kernel (measured), then storage/multiplication variants for the improved config
(computed from layouts / op counts, labelled analytical where they are).

In [ ]:
def fixedpoint_study(layer=12, L=512):
    mix = [m for m in TEACHER.modules() if isinstance(m, ShiftScanMixer)][layer]
    mix.cap = {"fx": True}; mix.decay = "nearest"
    x = CALIB_TOK[:L].reshape(1, L).long().to(DEV)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        TEACHER(input_ids=x, use_cache=False)
    cap, mix.cap, mix.decay = mix.cap, None, None
    dt, xh, Bh, Ch, dtA_q = [cap[k][0].double() for k in ("dt", "xh", "Bh", "Ch", "dtA_q")]   # drop batch dim
    a = torch.exp(dtA_q)                                                                       # (L,H) quantized decay
    k_int = torch.round(-torch.log2((1 - a).clamp(1e-12, 1))).clamp(0, KMAX).long()
    u = (dt[..., None, None] * xh[..., :, None] * Bh[..., None, :])                           # (L,H,P,N)
    h = torch.zeros_like(u[0]); ys = []
    for t in range(L):                                                                         # float64 reference
        h = a[t][:, None, None] * h + u[t]; ys.append((Ch[t][:, None, :] * h).sum(-1))
    y_ref = torch.stack(ys)
    rows = []
    for Fb in (12, 16, 20, 24):
        for mode in ("trunc", "round"):
            hi = torch.zeros(u.shape[1:], dtype=torch.int64); ys = []
            for t in range(L):
                k = k_int[t][:, None, None]
                if mode == "trunc": sh = hi >> k
                else: sh = torch.where(k > 0, (hi + (torch.ones_like(k) << (k - 1).clamp(min=0))) >> k, hi)
                hi = hi - sh + torch.round(u[t] * (2 ** Fb)).long()
                ys.append((Ch[t][:, None, :] * (hi.double() / 2 ** Fb)).sum(-1))
            y = torch.stack(ys)
            rows.append(dict(F_bits=Fb, shift=mode, rel_err=((y - y_ref).norm() / y_ref.norm()).item()))
    return pd.DataFrame(rows), np.percentile(k_int.numpy(), [10, 50, 90])

FX_DF, FX_K = fixedpoint_study()
print("[MEASURED] real layer-12 readout error of integer shift-scan vs float64 (same quantized decay); k percentiles 10/50/90 =", FX_K)
print(FX_DF.pivot(index="F_bits", columns="shift", values="rel_err").to_string())
FX_DF.to_csv(OUT / "phase5_fixedpoint.csv", index=False)

In [ ]:
def pack2(q):                                    # q int8 (N,K) in {-1,0,1} -> uint8 (N,K/4)
    c = torch.where(q == 1, 1, torch.where(q == -1, 2, 0)).to(torch.uint8).reshape(q.shape[0], -1, 4)
    return c[..., 0] | (c[..., 1] << 2) | (c[..., 2] << 4) | (c[..., 3] << 6)

def unpack2(pk, K):
    c = torch.stack([(pk >> (2 * i)) & 3 for i in range(4)], -1).reshape(pk.shape[0], -1)[:, :K].to(torch.int8)
    return torch.where(c == 1, 1, torch.where(c == 2, -1, 0)).to(torch.int8)

def pack_base3(q):                               # 5 trits per byte
    n = q.numel(); t = (q.reshape(-1) + 1).to(torch.int64)
    pad = (-n) % 5
    t = F.pad(t, (0, pad)).reshape(-1, 5)
    return (t * (3 ** torch.arange(5))).sum(-1).to(torch.uint8), n

def unpack_base3(pk, n):
    v = pk.to(torch.int64)[:, None]
    t = (v // (3 ** torch.arange(5))) % 3
    return (t.reshape(-1)[:n] - 1).to(torch.int8)

# ---- measured whole-model storage (using the best rule's real ternary patterns) ----
def storage_report(rule):
    tot = dict(fp16_total=0, tern_weights_2bit=0, tern_weights_b3=0, scales=0, fp_remainder=0, n_tern=0, n_all=0, zeros=0)
    mix = [m for m in TEACHER.modules() if isinstance(m, ShiftScanMixer)]
    for m in mix:
        for which, lin in (("in", m.in_proj), ("out", m.out_proj)):
            W = lin.weight.detach().float().cpu()
            if which == "in":
                keep = torch.ones(W.shape[0], dtype=torch.bool); keep[-m.H:] = False
                keep[2 * m.inter: 2 * m.inter + 2 * m.G * m.N] = False
                Wt, Wfp = W[keep], W[~keep]
            else:
                Wt, Wfp = W, W[:0]
            s = ACT_STATS[(m.layer_idx, which)] if "act" in rule else None
            q, alpha = ternary_parts(Wt, rule, GROUP, s)
            q8 = q.reshape(Wt.shape).to(torch.int8)
            pk = pack2(q8); assert torch.equal(unpack2(pk, Wt.shape[1]), q8), "2-bit pack roundtrip failed"
            pk3, n = pack_base3(q8); assert torch.equal(unpack_base3(pk3, n), q8.reshape(-1)), "base-3 roundtrip failed"
            tot["tern_weights_2bit"] += pk.numel(); tot["tern_weights_b3"] += pk3.numel()
            tot["scales"] += alpha.numel() * 2                      # fp16 scales
            tot["fp_remainder"] += Wfp.numel() * 2
            tot["n_tern"] += Wt.numel(); tot["zeros"] += (q8 == 0).sum().item()
    for n_, p_ in BASE.named_parameters():
        tot["fp16_total"] += p_.numel() * 2; tot["n_all"] += p_.numel()
    in_out = sum(m.in_proj.weight.numel() + m.out_proj.weight.numel() for m in mix)
    other_fp = tot["fp16_total"] - in_out * 2                        # embeddings/LM head, conv, norms, A/D/dt (kept fp16 here)
    tot["fp_remainder"] += other_fp
    return tot

ST = storage_report(BEST_RULE)
fp16 = ST["fp16_total"]
def ratio(extra_tern): return fp16 / (extra_tern + ST["scales"] + ST["fp_remainder"])
STORAGE_DF = pd.DataFrame([
    dict(kind="MEASURED (packed tensors)", scheme="FP16 baseline", bytes=fp16, compression_x=1.0),
    dict(kind="MEASURED (packed tensors)", scheme="ternary 2-bit + fp16 group scales, rest FP16", bytes=ST["tern_weights_2bit"] + ST["scales"] + ST["fp_remainder"], compression_x=ratio(ST["tern_weights_2bit"])),
    dict(kind="MEASURED (packed tensors)", scheme="ternary base-3 (1.6 bit) + scales, rest FP16", bytes=ST["tern_weights_b3"] + ST["scales"] + ST["fp_remainder"], compression_x=ratio(ST["tern_weights_b3"])),
])
tern_only_fp16 = ST["n_tern"] * 2
print(STORAGE_DF.to_string(index=False))
print(f"\nTernary-eligible weights are {ST['n_tern']/ST['n_all']:.1%} of all parameters; the rest "
      f"({ST['fp_remainder']/fp16:.1%} of FP16 bytes incl. embeddings/LM head) stays FP16, which caps whole-model compression (Amdahl).")
print(f"On the ternary matrices alone (measured): 2-bit {tern_only_fp16/(ST['tern_weights_2bit']+ST['scales']):.2f}x, "
      f"base-3 {tern_only_fp16/(ST['tern_weights_b3']+ST['scales']):.2f}x vs FP16. Measured zero fraction {ST['zeros']/ST['n_tern']:.2%}.")
STORAGE_DF.to_csv(OUT / "phase6_storage.csv", index=False)

# ---- Triton ternary GEMM vs cuBLAS FP16 ----
KERNEL_DF, KERNEL_OK = pd.DataFrame(), False
try:
    import triton, triton.language as tl

    @triton.jit
    def tern_gemm_kernel(x_ptr, w_ptr, s_ptr, o_ptr, M, N, K,
                         stride_xm, stride_wn, stride_sn, stride_om,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, GROUP_K: tl.constexpr):
        pid_m = tl.program_id(0); pid_n = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        offs_k = tl.arange(0, GROUP_K)
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
        for kg in range(0, K // GROUP_K):
            k0 = kg * GROUP_K
            x = tl.load(x_ptr + offs_m[:, None] * stride_xm + (k0 + offs_k)[None, :], mask=offs_m[:, None] < M, other=0.0)
            byte = tl.load(w_ptr + offs_n[:, None] * stride_wn + ((k0 + offs_k) // 4)[None, :],
                           mask=offs_n[:, None] < N, other=0).to(tl.int32)
            code = (byte >> (((offs_k % 4) * 2)[None, :])) & 3
            w = tl.where(code == 1, 1.0, tl.where(code == 2, -1.0, 0.0)).to(tl.float16)
            s = tl.load(s_ptr + offs_n * stride_sn + kg, mask=offs_n < N, other=0.0).to(tl.float32)
            acc += tl.dot(x, tl.trans(w)) * s[None, :]
        tl.store(o_ptr + offs_m[:, None] * stride_om + offs_n[None, :], acc.to(tl.float16),
                 mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

    def tern_gemm(x, pk, sc):
        M, K = x.shape; N = pk.shape[0]
        out = torch.empty((M, N), dtype=torch.float16, device=x.device)
        bm = 16 if M <= 16 else 64
        grid = (triton.cdiv(M, bm), triton.cdiv(N, 64))
        tern_gemm_kernel[grid](x, pk, sc, out, M, N, K, x.stride(0), pk.stride(0), sc.stride(0), out.stride(0),
                               BLOCK_M=bm, BLOCK_N=64, GROUP_K=GROUP, num_warps=4)
        return out

    def bench(fn, reps=50, warm=10):
        for _ in range(warm): fn()
        torch.cuda.synchronize(); ts = []
        for _ in range(reps):
            a_, b_ = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            a_.record(); fn(); b_.record(); torch.cuda.synchronize(); ts.append(a_.elapsed_time(b_))
        return float(np.median(ts))

    PEAK_BW = {"Tesla T4": 320.0, "Tesla P100-PCIE-16GB": 732.0}.get(torch.cuda.get_device_name(0))
    rows = []
    for (N, K, tag) in [(3072, 768, "in_proj (ternary rows)"), (768, 1536, "out_proj")]:
        Wf = torch.randn(N, K, device=DEV) * 0.05
        q, alpha = ternary_parts(Wf, BEST_RULE, GROUP, torch.ones(K, device=DEV) if "act" in BEST_RULE else None)
        q8 = q.reshape(N, K).to(torch.int8)
        pk = pack2(q8.cpu()).to(DEV); sc = alpha.reshape(N, K // GROUP).half().contiguous()
        Wq16 = (q * alpha).reshape(N, K).half(); W16 = Wf.half()
        for M in (1, 4, 16, 64, 512):
            x = torch.randn(M, K, device=DEV).half()
            ref = F.linear(x, Wq16)
            err = (tern_gemm(x, pk, sc).float() - ref.float()).abs().max().item() / ref.float().abs().max().item()
            t_fp16 = bench(lambda: F.linear(x, W16))
            t_tern = bench(lambda: tern_gemm(x, pk, sc))
            t_deq = bench(lambda: F.linear(x, (unpack2(pk, K).half().reshape(N, K // GROUP, GROUP) * sc[..., None]).reshape(N, K)))
            wb16, wbt = N * K * 2, pk.numel() + sc.numel() * 2
            rows.append(dict(layer=tag, M=M, fp16_ms=t_fp16, ternary_triton_ms=t_tern, dequant_then_cublas_ms=t_deq,
                             speedup_triton_vs_fp16=t_fp16 / t_tern, max_rel_err=err,
                             achieved_GBs_triton=wbt / (t_tern * 1e-3) / 1e9,
                             ANALYTICAL_bw_bound_speedup=(wb16 / wbt) if PEAK_BW else None))
    KERNEL_DF = pd.DataFrame(rows); KERNEL_OK = True
    print("[MEASURED latency; last column ANALYTICAL] ternary GEMM vs cuBLAS FP16 on", torch.cuda.get_device_name(0))
    print(KERNEL_DF.round(4).to_string(index=False))
    KERNEL_DF.to_csv(OUT / "phase6_kernel.csv", index=False)
except Exception as e:
    print("Triton benchmark could not run (reported as NOT ESTABLISHED):", repr(e)[:300])

In [ ]:
def compression_table():
    H, Pd, G, N, D = CFG.num_heads, CFG.head_dim, CFG.n_groups, CFG.state_size, CFG.hidden_size
    I = H * Pd; Lyr = CFG.num_hidden_layers; V = CFG.vocab_size
    fpset = set(BEST_Q.get("fp_proj", []))
    tern = scl = fpx = 0.0
    for li in range(Lyr):
        for w_ in ("in", "out"):
            n = proj_params(w_)
            if f"{li}:{w_}" in fpset: fpx += 2 * n
            else: tern += n / 4; scl += n / GROUP * 2
    all_tern = Lyr * (proj_params("in") + proj_params("out")) / 4
    if abs(all_tern - ST["tern_weights_2bit"]) > 1:
        print("WARNING: layout formula differs from measured packed size:", all_tern, ST["tern_weights_2bit"])
    rest = ST["fp_remainder"]; emb16 = V * D * 2
    rows = [("ternary 2-bit, all projections, FP16 embeddings", ST["tern_weights_2bit"] + ST["scales"] + rest),
            ("  + sensitive projections kept FP (chosen config)", tern + scl + fpx + rest),
            ("  + int8 embeddings/LM head (per-row scale)", tern + scl + fpx + rest - emb16 + V * D + V * 2),
            ("  + int4 embeddings/LM head (group-128)", tern + scl + fpx + rest - emb16 + V * D / 2 + V * D / GROUP * 2)]
    return pd.DataFrame([dict(scheme=s, bytes=int(b), compression_x=ST["fp16_total"] / b,
                              basis="packed layout sizes (2-bit case verified against measured tensors)") for s, b in rows])
COMP_DF = compression_table()
print(COMP_DF.round(3).to_string(index=False)); COMP_DF.to_csv(OUT / "stage4_compression_variants.csv", index=False)

In [ ]:
def mult_census(fp_rows="dt_bc"):
    H, Pd, G, N, D = CFG.num_heads, CFG.head_dim, CFG.n_groups, CFG.state_size, CFG.hidden_size
    I = H * Pd; conv = I + 2 * G * N; Lyr = CFG.num_hidden_layers; V = CFG.vocab_size
    in_rows = 2 * I + conv - I + H                     # gate + x + B + C + dt
    fp_r = {"none": 0, "dt": H, "dt_bc": H + 2 * G * N}[fp_rows]
    t_rows = in_rows - fp_r
    c = {}
    c["ternary in_proj adds"] = t_rows * D
    c["ternary out_proj adds"] = D * I
    c["ternary group-scale multiplies"] = t_rows * (D // GROUP) + D * (I // GROUP)
    c["FP in_proj rows (B,C,dt) MACs"] = fp_r * D
    c["conv1d MACs"] = conv * CFG.conv_kernel
    c["state decay (shift+sub, no mult)"] = H * Pd * N
    c["state inject dt*x*B multiplies"] = H * Pd * N + H * Pd
    c["readout C.h MACs"] = H * Pd * N
    c["D skip + gate/norm multiplies"] = I + 5 * I
    free = c["ternary in_proj adds"] + c["ternary out_proj adds"] + c["state decay (shift+sub, no mult)"]
    mult_layer = sum(v for k, v in c.items() if ("MAC" in k or "multiplies" in k))
    dense_layer = in_rows * D + D * I + c["conv1d MACs"] + 3 * H * Pd * N + 7 * I   # FP baseline: a*h is a multiply too
    out = dict(per_layer=c, mult_free_ops_layer=free, remaining_mult_layer=mult_layer, dense_mult_layer=dense_layer,
               lm_head_macs=V * D, layers=Lyr)
    out["model_remaining_mult"] = mult_layer * Lyr + V * D
    out["model_dense_mult"] = dense_layer * Lyr + V * D
    out["model_remaining_mult_wo_head"] = mult_layer * Lyr
    out["model_dense_mult_wo_head"] = dense_layer * Lyr
    return out

CEN = mult_census(FP_ROWS_DEFAULT)
CEN_DF = pd.DataFrame([dict(op=k, per_token_per_layer=v) for k, v in CEN["per_layer"].items()])
print("[ANALYTICAL] per-layer op census:"); print(CEN_DF.to_string(index=False))
print(f"\nLayers only : dense multiplies {CEN['model_dense_mult_wo_head']:,} -> remaining {CEN['model_remaining_mult_wo_head']:,} "
      f"({1 - CEN['model_remaining_mult_wo_head']/CEN['model_dense_mult_wo_head']:.1%} eliminated)")
print(f"Whole model : dense multiplies {CEN['model_dense_mult']:,} -> remaining {CEN['model_remaining_mult']:,} "
      f"({1 - CEN['model_remaining_mult']/CEN['model_dense_mult']:.1%} eliminated)")
print(f"LM head alone = {CEN['lm_head_macs']:,} MACs = {CEN['lm_head_macs']/CEN['model_remaining_mult']:.1%} of the remaining multiplies "
      f"(it stays FP unless it is also quantized). The SSM inject/readout products (dt*x*B, C.h) are the next largest.")
CEN_DF.to_csv(OUT / "phase7_mult_census.csv", index=False)

In [ ]:
def census_variants():
    base = mult_census(FP_ROWS_DEFAULT)
    H, G, N, D = CFG.num_heads, CFG.n_groups, CFG.state_size, CFG.hidden_size
    fp_r = {"none": 0, "dt": H, "dt_bc": H + 2 * G * N}[FP_ROWS_DEFAULT]
    Lyr, V = CFG.num_hidden_layers, CFG.vocab_size
    row_macs = fp_r * D; row_scale = fp_r * (D // GROUP)
    out = []
    for label, rows_t, head_t in (("as measured (FP rows + FP head)", False, False), ("all in_proj rows ternary", True, False),
                                   ("LM head ternary", False, True), ("rows + head ternary", True, True)):
        per_layer = base["remaining_mult_layer"] - (row_macs - row_scale if rows_t else 0)
        head = V * (D // GROUP) if head_t else V * D
        tot = per_layer * Lyr + head
        out.append(dict(variant=label, remaining_mult_per_token=int(tot), eliminated_pct=100 * (1 - tot / base["model_dense_mult"])))
    return pd.DataFrame(out)
CENV_DF = census_variants()
print("[ANALYTICAL] multiplications per generated token under design variants (per-group scale multiplies counted as multiplies):")
print(CENV_DF.round(2).to_string(index=False)); CENV_DF.to_csv(OUT / "stage4_census_variants.csv", index=False)

## Stage 5: longer training and quantized baselines
`v2_best_long` trains the chosen config for `long_tokens` (seed 0). `v2_int8` / `v2_int4` are group-128 QAT baselines on the same projections
(LR `INT_LR`, not tuned: report them as indicative). `v2_plain_twn` re-trains the plain v1 recipe with a checkpoint, so benchmarks can compare it against the improved model.

In [ ]:
if not SCALE_MODE:
    if DO["long"]:
        run_experiment("v2_best_long", BEST_Q, SEEDS[0], LONG_TOKENS, BEST_LR, save_ckpt=True)
    if DO["int_baselines"]:
        run_experiment("v2_int8", dict(intq=8, decay=None), SEEDS[0], TOKENS, INT_LR, save_ckpt=True)
        run_experiment("v2_int4", dict(intq=4, decay=None), SEEDS[0], TOKENS, INT_LR, save_ckpt=True)
    if DO["plain_ckpt"]:
        run_experiment("v2_plain_twn", dict(ternary=BASE_RULE, decay=None), SEEDS[0], TOKENS, LR_BASE, save_ckpt=True)
print(f"time used so far: {(time.time()-T0)/3600:.2f} h")

## Stage 6: extra benchmarks and low-bit embeddings / LM head
Own log-likelihood evaluator (length-normalised accuracy `acc_norm` like lm-eval) on HellaSwag, ARC-easy and PIQA, evenly sampled `bench_n` examples each.
Run for the teacher and every saved checkpoint. Datasets that cannot be loaded are skipped and reported. The embedding test fake-quantizes the (tied) embedding/LM head
of trained checkpoints post-hoc: no retraining.

In [ ]:
def _hs_prep(t):
    t = t.strip().replace(" [title]", ". ")
    return re.sub(r"\[.*?\]", "", t).replace("  ", " ")

def load_mc(task, n):
    if task == "hellaswag":
        ds = load_dataset("Rowan/hellaswag", split="validation")
        idx = np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
        out = []
        for i in idx:
            ex = ds[int(i)]
            q = _hs_prep(ex["activity_label"] + ": " + ex["ctx_a"] + " " + ex["ctx_b"].capitalize())
            out.append((q, [" " + _hs_prep(e) for e in ex["endings"]], int(ex["label"])))
        return out
    if task == "arc_easy":
        ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
        idx = np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
        out = []
        for i in idx:
            ex = ds[int(i)]
            out.append((f"Question: {ex['question']}\nAnswer:", [" " + t for t in ex["choices"]["text"]],
                        ex["choices"]["label"].index(ex["answerKey"])))
        return out
    if task == "piqa":
        last = None
        for rid in ("ybisk/piqa", "baber/piqa"):
            try:
                ds = load_dataset(rid, split="validation"); break
            except Exception as e:
                last = e; ds = None
        if ds is None: raise last
        idx = np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
        return [(f"Question: {ds[int(i)]['goal']}\nAnswer:", [" " + ds[int(i)]["sol1"], " " + ds[int(i)]["sol2"]], int(ds[int(i)]["label"])) for i in idx]
    raise ValueError(task)

@torch.no_grad()
def mc_eval(model, items):
    model.eval(); acc = accn = 0
    for q, choices, label in items:
        ci = TOK(q, add_special_tokens=False)["input_ids"][-900:]
        seqs, conts = [], []
        for c in choices:
            ti = TOK(c, add_special_tokens=False)["input_ids"]
            seqs.append(ci + ti); conts.append(ti)
        Lm = max(len(s) for s in seqs)
        ids = torch.zeros(len(seqs), Lm, dtype=torch.long)
        for i, s in enumerate(seqs): ids[i, :len(s)] = torch.tensor(s)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(input_ids=ids.to(DEV), use_cache=False).logits.float()
        lps = []
        for i, ti in enumerate(conts):
            lp = F.log_softmax(lg[i, len(ci) - 1: len(ci) + len(ti) - 1], -1)
            lps.append(lp.gather(-1, torch.tensor(ti, device=DEV)[:, None]).sum().item())
        acc += int(int(np.argmax(lps)) == label)
        accn += int(int(np.argmax([l_ / max(1, len(c.strip())) for l_, c in zip(lps, choices)])) == label)
    return acc / len(items), accn / len(items)

BENCH_ROWS, EMB_ROWS, ITEMS = [], [], {}
def fq_rows(W, bits, g=None):
    qmax = 2 ** (bits - 1) - 1
    if g is None:
        s = (W.abs().amax(-1, keepdim=True) / qmax).clamp_min(1e-8)
        return (W / s).round().clamp(-qmax, qmax) * s
    Wg = W.reshape(W.shape[0], -1, g)
    s = (Wg.abs().amax(-1, keepdim=True) / qmax).clamp_min(1e-8)
    return ((Wg / s).round().clamp(-qmax, qmax) * s).reshape(W.shape)

def emb_quant_eval(label, model):
    base_ppl = eval_ppl(model, EVAL_TOK)[0]
    EMB_ROWS.append(dict(model=label, embedding="fp32/fp16 (as trained)", fineweb_ppl=base_ppl, delta_pct=0.0))
    emb = model.backbone.embeddings.weight
    W0 = emb.data.clone()
    for tag, bits, g in (("int8 per-row", 8, None), ("int4 group-128", 4, GROUP)):
        emb.data = fq_rows(W0.float(), bits, g).to(W0.dtype)
        ppl = eval_ppl(model, EVAL_TOK)[0]
        EMB_ROWS.append(dict(model=label, embedding=tag, fineweb_ppl=ppl, delta_pct=100 * (ppl / base_ppl - 1)))
    emb.data = W0

if DO["bench"] or DO["embquant"]:
    if DO["bench"]:
        for t_ in ("hellaswag", "arc_easy", "piqa"):
            try:
                ITEMS[t_] = load_mc(t_, BENCH_N)
            except Exception as e:
                print(f"benchmark {t_} unavailable:", repr(e)[:120])
    def bench_model(label, model):
        row = {"model": label}
        for t_, items in ITEMS.items():
            a_, an_ = mc_eval(model, items); row[f"{t_}_acc"] = a_; row[f"{t_}_acc_norm"] = an_
        BENCH_ROWS.append(row); print(row)
    if DO["bench"] and ITEMS: bench_model("teacher", TEACHER)
    if DO["embquant"]: emb_quant_eval("teacher", TEACHER)
    for r_ in sorted(DONE.values(), key=lambda r: r["name"]):
        pth = r_.get("ckpt")
        if not pth or not Path(pth).exists() or time_left_h() < 0.2: continue
        label = f"{r_['name']} (s{r_['seed']}, {r_['tokens']//1_000_000}M tok)"
        mdl = load_student(r_["qcfg"], pth)
        if DO["bench"] and ITEMS: bench_model(label, mdl)
        if DO["embquant"] and r_["name"] in ("v2_best", "v2_best_long"): emb_quant_eval(label, mdl)
        del mdl; gc.collect(); torch.cuda.empty_cache()
    BENCH_DF, EMB_DF = pd.DataFrame(BENCH_ROWS), pd.DataFrame(EMB_ROWS)
    if len(BENCH_DF): print("\n[MEASURED] benchmarks (n=%d per task):" % BENCH_N); print(BENCH_DF.round(3).to_string(index=False)); BENCH_DF.to_csv(OUT / "stage6_benchmarks.csv", index=False)
    if len(EMB_DF): print("\n[MEASURED] embedding/LM-head quantization, no retraining:"); print(EMB_DF.round(3).to_string(index=False)); EMB_DF.to_csv(OUT / "stage6_embedding_quant.csv", index=False)
else:
    BENCH_DF, EMB_DF = pd.DataFrame(), pd.DataFrame()

## Stage 7: statistics, report, automatic claims checklist
v1 baselines come from the attached `runs.jsonl` (hash differs by design; both hashes are printed). Pairing is by seed with identical data order.

In [ ]:
def recs(name, tokens=None):
    return [r for r in DONE.values() if r["name"] == name and (tokens is None or r["tokens"] == tokens)
            and r.get("metrics") and r["metrics"].get("fineweb_ppl")]

GROUPS = [("v1 fp_student (no quantizer)", "fp_student", TOKENS), ("v1 plain ternary (twn)", "tern_twn", TOKENS),
          ("v2 plain ternary rerun", "v2_plain_twn", TOKENS), ("v2 improved ternary", "v2_best", TOKENS)] \
         + [(f"v2 improved + {m} decay", f"v2_best_{m}", TOKENS) for m in FULL_DECAYS] \
         + [("v2 improved, long run", "v2_best_long", LONG_TOKENS), ("v2 int8 QAT baseline", "v2_int8", TOKENS), ("v2 int4 QAT baseline", "v2_int4", TOKENS),
            (f"scale {MODEL_SIZE} plain", f"scale_{MODEL_SIZE}_plain", SCALE_TOKENS), (f"scale {MODEL_SIZE} best", f"scale_{MODEL_SIZE}_best", SCALE_TOKENS)]
TROWS = []
for label, name, tok in GROUPS:
    rs = recs(name, tok)
    if not rs: continue
    v = np.array([r["metrics"]["fineweb_ppl"] for r in rs])
    def mean_of(k):
        x = [r["metrics"][k] for r in rs if r["metrics"].get(k) is not None]
        return float(np.mean(x)) if x else None
    lg = [r["metrics"]["long_gap"][-1] for r in rs if r["metrics"].get("long_gap")]
    TROWS.append(dict(config=label, n_seeds=len(rs), tokens=tok, ppl_mean=v.mean(), ppl_std=(v.std(ddof=1) if len(v) > 1 else float("nan")),
                      ratio_vs_teacher=v.mean() / T_PPL, wiki_ppl=mean_of("wiki_ppl"), lambada=mean_of("lambada_acc"),
                      late_ctx_nll_gap=(float(np.mean(lg)) if lg else None), code_hash=",".join(sorted({r["code_hash"] for r in rs}))))
TABLE = pd.DataFrame(TROWS)
print(f"teacher: FineWeb PPL {T_PPL:.3f}, WikiText PPL {TEACHER_METRICS.get('wiki_ppl')}, LAMBADA {TEACHER_METRICS.get('lambada_acc')}\n")
print(TABLE.round(3).to_string(index=False)); TABLE.to_csv(OUT / "stage7_summary.csv", index=False)

def paired_vs(a_name, b_name, tok=TOKENS, nboot=10000):
    A = {r["seed"]: r["metrics"]["fineweb_ppl"] for r in recs(a_name, tok)}
    B = {r["seed"]: r["metrics"]["fineweb_ppl"] for r in recs(b_name, tok)}
    s = sorted(set(A) & set(B))
    if len(s) < 2: return None
    x, y = np.array([A[i] for i in s]), np.array([B[i] for i in s])
    d = np.log(x) - np.log(y); rng = np.random.default_rng(0)
    boots = [rng.choice(d, len(d)).mean() for _ in range(nboot)]
    return dict(a=a_name, b=b_name, n=len(s), mean_logppl_diff=d.mean(), ci_lo=np.percentile(boots, 2.5), ci_hi=np.percentile(boots, 97.5),
                ppl_ratio=float(np.exp(d.mean())), p_ttest=sst.ttest_rel(x, y).pvalue, gap_over_seed_std=abs(d.mean()) / (d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan))
PAIRS = [r for r in (paired_vs("v2_best", "tern_twn"), paired_vs("v2_best", "v2_plain_twn")) if r]
PAIR_DF = pd.DataFrame(PAIRS)
if len(PAIR_DF):
    print("\n[paired, same seeds/data order] mean_logppl_diff < 0 means the first (improved) config is better:")
    print(PAIR_DF.round(4).to_string(index=False)); PAIR_DF.to_csv(OUT / "stage7_paired.csv", index=False)
else:
    print("\nNo paired comparison possible (need >=2 matched seeds of v2_best and tern_twn; attach the v1 runs.jsonl as input).")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for i, r_ in TABLE.iterrows():
    ax[0].bar(i, r_.ppl_mean, alpha=.6)
ax[0].set_xticks(range(len(TABLE))); ax[0].set_xticklabels(TABLE.config, rotation=70, ha="right", fontsize=7)
ax[0].axhline(T_PPL, color="r", ls="--", label="teacher"); ax[0].set_yscale("log"); ax[0].set_ylabel("FineWeb-Edu PPL"); ax[0].legend(); ax[0].set_title("Mean student PPL")
for nm, lab in (("tern_twn", "v1 plain twn"), ("v2_best", "v2 improved")):
    rr = [r for r in DONE.values() if r["name"] == nm and r["seed"] == SEEDS[0] and r["tokens"] == TOKENS and r["loss_curve"]]
    if rr:
        c_ = np.array(rr[0]["loss_curve"]); ax[1].plot(c_[:, 0], c_[:, 1], label=lab)
ax[1].set_yscale("log"); ax[1].set_xlabel("step"); ax[1].set_ylabel("train KD"); ax[1].legend(); ax[1].set_title("Loss curves (seed 0)")
for nm, lab in (("tern_twn", "v1 plain twn"), ("v2_best", "v2 improved"), ("v2_best_long", "v2 long")):
    rr = [r for r in DONE.values() if r["name"] == nm and r.get("metrics") and r["metrics"].get("long_gap")]
    if rr: ax[2].plot(np.mean([r["metrics"]["long_gap"] for r in rr], 0), marker="o", label=lab)
ax[2].axhline(0, color="k", lw=.5); ax[2].set_xlabel("position bucket"); ax[2].set_ylabel("NLL(student) - NLL(teacher)"); ax[2].legend(); ax[2].set_title("Long-context drift")
plt.tight_layout(); plt.savefig(OUT / "results_overview_v2.png", dpi=140); plt.show()

In [ ]:
def row_of(name, tok):
    rs = recs(name, tok)
    return float(np.mean([r["metrics"]["fineweb_ppl"] for r in rs])) / T_PPL if rs else float("nan")
def st(cond, yes, no): return ("ESTABLISHED: " + yes) if cond else ("NOT ESTABLISHED: " + no)

r_plain, r_best, r_long = row_of("tern_twn", TOKENS), row_of("v2_best", TOKENS), row_of("v2_best_long", LONG_TOKENS)
r_int8, r_int4 = row_of("v2_int8", TOKENS), row_of("v2_int4", TOKENS)
best_any = np.nanmin([r_best, r_long]) if not (np.isnan(r_best) and np.isnan(r_long)) else float("nan")
pv = PAIRS[0] if PAIRS else None
kern = KERNEL_DF.speedup_triton_vs_fp16.max() if (KERNEL_OK and len(KERNEL_DF)) else float("nan")
emb8 = EMB_DF[(EMB_DF.embedding == "int8 per-row") & EMB_DF.model.str.startswith("v2_best")].delta_pct.mean() if len(EMB_DF) else float("nan")
rows = [
    ("Improved recipe beats plain twn (paired, same seeds & data)", st(pv is not None and pv["mean_logppl_diff"] < 0 and pv["ci_hi"] < 0,
        f"PPL ratio {pv['ppl_ratio']:.3f} (95% CI on log diff [{pv['ci_lo']:.3f}, {pv['ci_hi']:.3f}])" if pv else "",
        (f"ratio {pv['ppl_ratio']:.3f}, CI [{pv['ci_lo']:.3f}, {pv['ci_hi']:.3f}] includes 0 or worse" if pv else "no paired data (need v1 runs attached and >=2 seeds)"))),
    ("Best ternary student within 1.3x of teacher PPL", st(best_any <= 1.3, f"{best_any:.2f}x", f"best is {best_any:.2f}x (v1 plain twn was {r_plain:.2f}x)")),
    ("Longer training helps (long run better than 4M run)", st(r_long < r_best, f"{r_long:.2f}x vs {r_best:.2f}x", f"{r_long:.2f}x vs {r_best:.2f}x or not run")),
    ("Ternary vs int8/int4 QAT baselines", f"measured: ternary {r_best:.2f}x, int8 {r_int8:.2f}x, int4 {r_int4:.2f}x teacher PPL (int LR untuned)"),
    ("8-bit embeddings/LM head cost (no retraining)", st(np.isfinite(emb8) and abs(emb8) < 1.0, f"{emb8:+.2f}% PPL", f"{emb8:+.2f}% PPL or not run")),
    ("Whole-model compression incl. int8 embeddings", f"{COMP_DF.compression_x.iloc[2]:.2f}x (layout-computed; 2-bit ternary + int8 emb, chosen FP projections)"),
    ("Ternary kernel faster than cuBLAS FP16 at some batch size", st(np.isfinite(kern) and kern >= 1.1, f"best {kern:.2f}x", f"best measured {kern:.2f}x" if np.isfinite(kern) else "benchmark did not run")),
    ("Multiplication-free", "NOT ESTABLISHED: use 'reduced-multiplication'. Analytical variants: " + "; ".join(f"{r.variant}: {r.eliminated_pct:.1f}% eliminated" for r in CENV_DF.itertuples())),
    ("Extra benchmarks (HellaSwag/ARC-e/PIQA)", "measured for: " + (", ".join(BENCH_DF.model) if len(BENCH_DF) else "none (datasets unavailable or stage skipped)")),
    ("Second model size", "ran: " + MODEL_SIZE if SCALE_MODE else "NOT RUN (set MODEL_SIZE='370m' in a separate session; it reuses best_config.json)"),
]
REPORT = ["# Shift-Scan v2 report", "", f"Generated {time.strftime('%Y-%m-%d %H:%M:%S')} | preset `{PRESET}` | code `{CODE_VERSION}` hash `{CODE_HASH}` | {ENV['device']} | model `{MODEL_ID}`",
          f"Teacher FineWeb PPL {T_PPL:.3f} | chosen config `{json.dumps(BEST_Q)}` lr {BEST_LR:g} | v1 records are from a different code hash by design.", "",
          "## Claims checklist (auto-computed)", "", "_Note: with 3 seeds the bootstrap interval comes from 3 pairs and is optimistic; read it as indicative._", ""] + [f"- **{a}**: {b}" for a, b in rows]
REPORT += ["", "## Summary table", "", TABLE.round(3).to_markdown(index=False)]
if len(PAIR_DF): REPORT += ["", "## Paired comparison (log-PPL)", "", PAIR_DF.round(4).to_markdown(index=False)]
if len(SENS_DF): REPORT += ["", "## Stage 1 sensitivity (top 12)", "", SENS_DF.head(12).round(4).to_markdown(index=False), "", f"Kept FP: {SENS_FP}"]
if ABL: REPORT += ["", "## Stage 2 short ablation", "", ABL_DF.round(4).to_markdown(index=False)]
REPORT += ["", "## Storage variants (layout-computed)", "", COMP_DF.round(3).to_markdown(index=False),
           "", "## Multiplication variants (analytical)", "", CENV_DF.round(2).to_markdown(index=False)]
if len(BENCH_DF): REPORT += ["", "## Benchmarks (measured)", "", BENCH_DF.round(3).to_markdown(index=False)]
if len(EMB_DF): REPORT += ["", "## Embedding quantization (measured)", "", EMB_DF.round(3).to_markdown(index=False)]
if KERNEL_OK: REPORT += ["", "## Kernel (measured; last column analytical)", "", KERNEL_DF.round(4).to_markdown(index=False)]
(OUT / "FINAL_REPORT_V2.md").write_text("\n".join(REPORT))
print("\n".join(f"- {a}: {b}" for a, b in rows))
print(f"\nAll outputs in {OUT}. Total time {(time.time()-T0)/3600:.2f} h. Save Version / download runs.jsonl + best_config.json now.")